# Predição dos modelos MSOV1 e MSOV2 nas Imagens de Mergulho

Autor:  Viviane da Rosa Sommer

Atualizado: 28/01/2025

## Objetivo:
Este notebook tem como objetivo mostrar as imagens originais de mergulho, juntamente com sua predições dos modelos MSOV1 e MSOV2.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import glob
import cv2
import numpy as np
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import traceback 
import torchvision
import torch
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.patches as patches 
from PIL import Image 
from matplotlib.path import Path 
from sklearn.metrics import auc 
from typing import List, Dict, Any, Tuple, Optional

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
mso_v1 = YOLO('../Coral_Note/runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')
mso_v2 = YOLO('../Testes-Segmentacao-PreProcessamento/runs/segment/train3/weights/best.pt')

INPUT_DIRECTORY_POSITIVE_IMAGES = "Dataset-Segm-Mergulho/original_images"
all_images = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.[pPjJ][nNpP][gGgG]") 
all_labels = [1] * len(all_images)

## Funções Necessárias

In [ ]:
def read_image_with_annotation(filename: str) -> Tuple[np.ndarray, Optional[np.ndarray]]: 
    """ 
    Reads an image and its corresponding annotation mask if available. 
 
    Args: 
        filename (str): Path to the image file. 
 
    Returns: 
        Tuple[np.ndarray, Optional[np.ndarray]]: A tuple containing: 
            - The image as a numpy array. 
            - The annotation mask as a numpy array, or None if no annotation is found. 
 
    Raises: 
        ValueError: If the image cannot be loaded. 
    """ 
    
    image = cv2.imread(filename) 
    if image is None: 
        raise ValueError(f"Unable to load image: {filename}") 
 
    txt_file = filename.replace('.jpg', '.txt').replace('.png', '.txt').replace('.JPG', '.txt').replace("original_images", "yolo_annotation") 
    annotation_mask = None 
    if os.path.exists(txt_file): 
        annotation_mask = load_annotations(txt_file, image.shape[:2]) 
    else: 
        print("No annotation file found.") 
    return image, annotation_mask 


def process_results(results: List[Any]) -> Optional[Any]: 
    """ 
    Extracts the first valid result containing masks from the model's output. 
 
    Args: 
        results (List[Any]): List of model detection results. 
 
    Returns: 
        Optional[Any]: The first valid result with masks, or None if no valid result exists. 
    """ 
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.ndarray, model: Any) -> np.ndarray: 
    """ 
    Predicts coral regions in an image using a given model. 
 
    Args: 
        img (np.ndarray): Input image. 
        model (Any): Model to use for prediction. 
 
    Returns: 
        np.ndarray: Binary mask for coral regions. 
    """ 
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    if coral_best_result and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_masks) > 0:
            nms_indices = torchvision.ops.nms(boxes[coral_indices, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int().cpu().numpy() * 255
        else:
            final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    else:
        final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)

    return cv2.resize(final_coral_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)


def generate_predictions(files: List[str], model_v1: Any, model_v2: Any) -> List[Dict[str, Any]]: 
    """ 
    Generates predictions for a list of image files using two models. 
 
    Args: 
        files (List[str]): List of file paths to process. 
        model_v1 (Any): First model for prediction. 
        model_v2 (Any): Second model for prediction. 
 
    Returns: 
        List[Dict[str, Any]]: A list of dictionaries containing results for each file: 
            - file (str): Original filename 
            - cropped_image (np.ndarray): The cropped input image 
            - annotation_mask (np.ndarray): The annotation mask (if available) 
            - mask_v1 (np.ndarray): Prediction mask from model_v1 
            - mask_v2 (np.ndarray): Prediction mask from model_v2 
            - iou_v1 (float): IoU score for model_v1 
            - iou_v2 (float): IoU score for model_v2 
            - dice_v1 (float): Dice score for model_v1 
            - dice_v2 (float): Dice score for model_v2 
 
    Note: 
        Errors during processing of individual files are caught and logged, 
        allowing the function to continue processing other files. 
    """ 
    
    results = [] 
    for filename in files: 
        try: 
            cropped_image, annotation_mask = read_image_with_annotation(filename) 
             
            mask_v1_result = prediction_coral(cropped_image, model_v1) 
             
            mask_v2_result = prediction_coral(cropped_image, model_v2) 
             
            if isinstance(mask_v1_result, tuple): 
                mask_v1 = mask_v1_result[0] 
            else: 
                mask_v1 = mask_v1_result 
             
            if isinstance(mask_v2_result, tuple): 
                mask_v2 = mask_v2_result[0] 
            else: 
                mask_v2 = mask_v2_result 
             
 
            iou_v1 = calculate_iou(mask_v1, annotation_mask) 
            iou_v2 = calculate_iou(mask_v2, annotation_mask) 
 
            dice_v1 = calculate_dice(mask_v1, annotation_mask) 
            dice_v2 = calculate_dice(mask_v2, annotation_mask) 
 
            results.append({ 
                "file": filename, 
                "cropped_image": cropped_image, 
                "annotation_mask": annotation_mask, 
                "mask_v1": mask_v1, 
                "mask_v2": mask_v2, 
                "iou_v1": iou_v1, 
                "iou_v2": iou_v2, 
                "dice_v1": dice_v1, 
                "dice_v2": dice_v2 
            }) 
        except Exception as e: 
            print(f"Error processing {filename}: {e}") 
            traceback.print_exc() 
    return results 


def load_annotations(annotation_path: str, image_size: Tuple[int, int]) -> np.ndarray: 
    """ 
    Loads annotations from a file and creates a binary mask. 
 
    Args: 
        annotation_path (str): Path to the annotation file. 
        image_size (Tuple[int, int]): Size of the image (height, width). 
 
    Returns: 
        np.ndarray: A binary mask where annotated regions are marked with 255. 
 
    Raises: 
        Exception: If there's an error reading the annotation file, it's caught, 
                   logged, and an empty mask is returned. 
 
    Note: 
        The annotation file is expected to contain polygon coordinates in YOLO format 
        (normalized coordinates). These are scaled to image dimensions and used to 
        create a binary mask. 
    """ 
    
    try: 
        mask = np.zeros(image_size, dtype=np.uint8) 
        with open(annotation_path, 'r') as file: 
            for line in file: 
                parts = line.strip().split() 
                points = np.array(list(map(float, parts[1:]))) 
                points = points.reshape(-1, 2) 
 
                points[:, 0] *= image_size[1] 
                points[:, 1] *= image_size[0]
 
                cv2.fillPoly(mask, [points.astype(np.int32)], 255) 
 
        return mask 
    except Exception as e: 
        print(f"Error reading annotation file {annotation_path}: {e}") 
        traceback.print_exc() 
        return np.zeros(image_size, dtype=np.uint8) 


def plot_image_comparisons(results: List[Dict[str, Any]]) -> None: 
    """ 
    Plots image comparisons for expert annotations and model predictions. 
 
    Args: 
        results (List[Dict[str, Any]]): A list of dictionaries containing results for each image. 
            Each dictionary should have the following keys: 
            - 'file': str, path to the image file 
            - 'mask_v1': np.ndarray, prediction mask from model v1 
            - 'mask_v2': np.ndarray, prediction mask from model v2 
 
    Returns: 
        None 
 
    This function creates a figure with three subplots for each image: 
    1. Original image with expert annotation (if available) 
    2. Original image with model v1 prediction 
    3. Original image with model v2 prediction 
 
    Contours are drawn around the annotated/predicted areas. 
    If an error occurs while processing an image, it's logged and the function continues with the next image. 
    """ 
    
    for result in results: 
        try: 
            image_path = result["file"] 
            mask_v1 = result["mask_v1"] 
            mask_v2 = result["mask_v2"] 
             
            img = cv2.imread(image_path) 
            if img is None: 
                print(f"Unable to load image: {image_path}") 
                continue 
             
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
            label = "Positive" 
             
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 10)) 
             
            original_image = Image.open(image_path) 
            original_size = original_image.size 
             
            ax1.imshow(original_image) 
            ax1.axis('off') 
            ax1.set_title('Expert Annotation', fontsize=14) 
 
            annotation_path = os.path.join("Dataset-Segm-Mergulho/yolo_annotation",     
                                           os.path.splitext(os.path.basename(image_path))[0] + ".txt") 
 
            def plot_mask(ax, mask, color, title): 
                ax.imshow(img) 
                ax.axis('off') 
                ax.set_title(f'{title}', fontsize=14) 
 
                binary_mask = (mask > 0.5).astype(np.uint8) 
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) 
                 
                for contour in contours: 
                    path = Path(contour[:, 0, :], closed=True) 
                    patch = patches.PathPatch(path, edgecolor=color, facecolor='none', lw=4) 
                    ax.add_patch(patch) 
 
            if os.path.exists(annotation_path): 
                expert_mask = load_annotations(annotation_path, original_size[::-1]) 
                plot_mask(ax1, expert_mask, 'r', 'Expert Annotation') 
            else: 
                print(f"No annotations found for {image_path}") 
 
            plot_mask(ax2, mask_v1, 'b', 'Model mso_v1 Prediction') 
            plot_mask(ax3, mask_v2, 'm', 'Model mso_v2 Prediction')  
 
            main_title = os.path.splitext(os.path.basename(image_path))[0] 
            plt.suptitle(f"{main_title} ({label})", fontsize=16) 
 
            plt.tight_layout() 
            plt.show() 
 
        except Exception as e: 
            print(f"Error plotting for {result['file']}: {e}") 
            traceback.print_exc() 
            continue 


def evaluate_predictions(results: List[Dict[str, Any]], labels: List[int]) -> None: 
    """ 
    Evaluates predictions using confusion matrices and classification reports. 
 
    Args: 
        results (List[Dict[str, Any]]): List of dictionaries containing predictions and annotations. 
        labels (List[int]): Ground truth labels for each image. 
 
    Returns: 
        None: Displays evaluation results. 
    """ 
    predictions_v1 = [1 if result["mask_v1"].any() else 0 for result in results] 
    predictions_v2 = [1 if result["mask_v2"].any() else 0 for result in results] 
 
    conf_matrix_v1 = confusion_matrix(labels, predictions_v1) 
    conf_matrix_v2 = confusion_matrix(labels, predictions_v2) 
 
    print("Classification Report for MSO_V1:") 
    print(classification_report(labels, predictions_v1, target_names=["Negative", "Positive"], zero_division=0)) 
 
    print("Classification Report for MSO_V2:") 
    print(classification_report(labels, predictions_v2, target_names=["Negative", "Positive"], zero_division=0)) 
 
    plot_confusion_matrices_side_by_side( 
        conf_matrices=[conf_matrix_v1, conf_matrix_v2], 
        model_names=["mso_v1", "mso_v2"], 
        class_names=["Negative", "Positive"] 
    ) 


def plot_confusion_matrices_side_by_side(conf_matrices: List[np.ndarray], model_names: List[str], class_names: List[str]) -> None: 
    """
    Plots multiple confusion matrices side by side.

    Args:
        conf_matrices (list): List of confusion matrices (numpy arrays).
        model_names (list): List of model names corresponding to each confusion matrix.
        class_names (list): List of class names (e.g., ["Negative", "Positive"]).

    Returns:
        None: Displays the confusion matrices in a single figure.
    """
    num_matrices = len(conf_matrices)
    plt.figure(figsize=(6 * num_matrices, 6))

    for i, (conf_matrix, model_name) in enumerate(zip(conf_matrices, model_names)):
        plt.subplot(1, num_matrices, i + 1)
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")

    plt.tight_layout()
    plt.show()


def calculate_iou(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float: 
    """
    Calculates the Intersection over Union (IoU) between a predicted mask and an annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: IoU value, or NaN if masks are invalid.
    """
    if annotation_mask is None or predicted_mask is None:
        return float('nan')  

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()
    union = np.logical_or(predicted_mask, annotation_mask).sum()

    if union == 0:
        return 1.0  

    iou = intersection / union
    return iou


def calculate_dice(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float: 
    """
    Calculates the Dice Similarity Coefficient (DICE) between the predicted mask and the annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: DICE value, or NaN if masks are invalid.
    """
    if predicted_mask is None or annotation_mask is None:
        return float('nan')  

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    predicted_sum = predicted_mask.sum()
    annotation_sum = annotation_mask.sum()

    if predicted_sum == 0 and annotation_sum == 0:
        return 1.0  

    if annotation_sum == 0 and predicted_sum > 0:
        return 0.0  

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()

    dice = (2 * intersection) / (predicted_sum + annotation_sum)
    return dice


def display_statistics(results: List[Dict[str, Any]]) -> None: 
    """
    Displays statistics (mean, median, etc.) for IoU and DICE predictions.

    Args:
        results (list): List of dictionaries containing IoU and DICE values.

    Returns:
        None: Prints the statistics.
    """
    iou_v1_values = [result["iou_v1"] for result in results if not np.isnan(result["iou_v1"])]
    iou_v2_values = [result["iou_v2"] for result in results if not np.isnan(result["iou_v2"])]

    dice_v1_values = [result["dice_v1"] for result in results if not np.isnan(result["dice_v1"])]
    dice_v2_values = [result["dice_v2"] for result in results if not np.isnan(result["dice_v2"])]

    print("IoU Statistics for MSO_V1:")
    print(f"Mean IoU: {np.mean(iou_v1_values):.4f}")
    print(f"Median IoU: {np.median(iou_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v1_values):.4f}")

    print("\nIoU Statistics for MSO_V2:")
    print(f"Mean IoU: {np.mean(iou_v2_values):.4f}")
    print(f"Median IoU: {np.median(iou_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v2_values):.4f}")

    print("\nDICE Statistics for MSO_V1:")
    print(f"Mean DICE: {np.mean(dice_v1_values):.4f}")
    print(f"Median DICE: {np.median(dice_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v1_values):.4f}")

    print("\nDICE Statistics for MSO_V2:")
    print(f"Mean DICE: {np.mean(dice_v2_values):.4f}")
    print(f"Median DICE: {np.median(dice_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v2_values):.4f}")


def calculate_map(results: List[Dict[str, Any]], iou_thresholds: List[float] = [0.5]) -> Dict[str, Dict[str, float]]: 
    """ 
    Calculates the Mean Average Precision (mAP) for the predictions of each model at specified IoU thresholds. 
 
    Args: 
        results (List[Dict[str, Any]]): List of dictionaries containing predictions and annotations. 
        iou_thresholds (List[float]): List of IoU thresholds to consider for mAP calculation. 
 
    Returns: 
        Dict[str, Dict[str, float]]: Dictionary containing mAP values for each model at specified IoU thresholds. 
    """ 
    def compute_precision_recall(predictions, annotations, scores, iou_threshold): 
        precisions = [] 
        recalls = [] 
        total_positives = sum(1 for mask in annotations if mask is not None and mask.sum() > 0) 
 
        for threshold in np.linspace(0, 1, num=101):   
            true_positives = 0 
            false_positives = 0 
             
            for pred_mask, annotation_mask, score in zip(predictions, annotations, scores): 
                if annotation_mask is None or annotation_mask.sum() == 0: 
                    if pred_mask.sum() > 0 and score >= threshold: 
                        false_positives += 1 
                    continue 
                 
                iou = calculate_iou(pred_mask, annotation_mask) 
                if iou >= iou_threshold and score >= threshold: 
                    true_positives += 1 
                elif score >= threshold: 
                    false_positives += 1 
             
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0 
            recall = true_positives / total_positives if total_positives > 0 else 0 
 
            precisions.append(precision) 
            recalls.append(recall) 
 
        if not precisions or not recalls: 
            return [0], [0] 
 
        return precisions, recalls 
 
    predictions_v1 = [result.get("mask_v1", np.zeros_like(result.get("annotation_mask", np.array([])))) for result in results] 
    predictions_v2 = [result.get("mask_v2", np.zeros_like(result.get("annotation_mask", np.array([])))) for result in results] 
    annotations = [result.get("annotation_mask", None) for result in results] 
 
    scores_v1 = [0.9] * len(results)   
    scores_v2 = [0.9] * len(results)   
 
    map_values = {"mAP_v1": {}, "mAP_v2": {}} 
 
    for iou_threshold in iou_thresholds: 
        precisions_v1, recalls_v1 = compute_precision_recall(predictions_v1, annotations, scores_v1, iou_threshold) 
        precisions_v2, recalls_v2 = compute_precision_recall(predictions_v2, annotations, scores_v2, iou_threshold) 
 
        recalls_v1, precisions_v1 = zip(*sorted(zip(recalls_v1, precisions_v1))) 
        recalls_v2, precisions_v2 = zip(*sorted(zip(recalls_v2, precisions_v2))) 
 
        map_v1 = auc(recalls_v1, precisions_v1) 
        map_v2 = auc(recalls_v2, precisions_v2) 
 
        map_values["mAP_v1"][f"mAP@{int(iou_threshold * 100)}"] = map_v1 
        map_values["mAP_v2"][f"mAP@{int(iou_threshold * 100)}"] = map_v2 
 
    return map_values 

## Visualização das predições

In [ ]:
results = generate_predictions(all_images, mso_v1, mso_v2)
plot_image_comparisons(results)

## Avaliação das métricas

In [ ]:
evaluate_predictions(results, all_labels)
display_statistics(results)

map_results = calculate_map(results, iou_thresholds=[0.5, 0.9])
print("\nmAP Results:")
for threshold, value in map_results["mAP_v1"].items():
    print(f"MSO_V1 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v2"].items():
    print(f"MSO_V2 - {threshold}: {value:.4f}")

In [ ]:
!jupyter nbconvert --to html Pred_Mergulho_MSOV1_MSOV2.ipynb